# LECTURE 2: DATA CLEANING (WEEK 3)
---
## Data integrity, outliers, missing values, and leakage-safe pipelines

In this lecture, we will explore the foundational steps of data preparation for machine learning in finance. A machine-learning project is not a straight line. Business understanding, data understanding, preparation, modeling, evaluation, and deployment continually inform one another. Model errors often send us back to preparation to revise cleaning, transformations, or features. Data preparation is the critical bridge between collected raw data and credible business insights.

# Part 1: Foundations of Data Preparation
---
The preparation process involves five main task groups:
1. **Data Cleaning**: Improves validity by correcting invalid, redundant, extreme, or missing observations.
2. **Feature Selection**: Removes unnecessary inputs to retain only informative predictors.
3. **Data Transforms**: Changes scale, distribution, or representation.
4. **Feature Engineering**: Creates new features to represent relationships more directly.
5. **Dimensionality Reduction**: Compresses many variables into fewer informative dimensions.

This notebook focuses entirely on **Data Cleaning**.


In [ ]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries loaded successfully.")


## 1.1 The Golden Rule: Data Leakage

### 1. Overview
Data leakage occurs when information from the test set (the "future") enters the model-training process (the "past"). Leakage can make validation performance look excellent while producing a model that is completely useless in practice. The Golden Rule of machine learning preparation is: **SPLIT FIRST, PREPARE LATER.**

### 2. Concept & Formula
Leakage happens during preparation when:
- Test statistics (like mean or variance of the entire dataset) influence scaling or imputation.
- Future information appears in historical predictors.
- Duplicate entities occur in both training and testing data.

**Correct Process**:
1. Split raw data first into $X_{train}$ and $X_{test}$.
2. **Fit** preparation parameters only on training data (e.g., finding the mean of $X_{train}$).
3. **Transform** both training and test data using the fitted object.

### 3. Simple Explanation
Imagine you are a student preparing for a final exam. The practice questions are your training data, and the final exam is the test data. Data leakage is like peeking at the final exam's answer key to help you study. You will score 100% on your practice tests, but when you face real-world problems later, you will fail because you only memorized the answers instead of learning the concepts.

### 4. Manual Calculation: How Leakage Changes Imputation
Let's see exactly how leakage distorts data mathematically.

**Incorrect: Fit before splitting**
Suppose your incomes are `(20, 30, ?)` for training and `(100, ?)` for testing (in thousands USD).
If you calculate the mean using ALL observed values:
$$ \bar{x}_{all} = \frac{20 + 30 + 100}{3} = 50 $$
You impute the missing values with 50. The test value `100` has improperly influenced the preprocessing learned for training! The leaked value 50 makes the training transformation unrealistically aware of future data (pulling the mean artificially high).

**Correct: Split, then fit on training**
Learn the imputation value ONLY from observed training values:
$$ \bar{x}_{train} = \frac{20 + 30}{2} = 25 $$
We fill both training and test missing values with `25`, keeping the test information completely isolated.

### 5. Code Python


In [ ]:
# Simulating Leakage vs No Leakage
train_incomes = [20, 30, np.nan]
test_incomes = [100, np.nan]
all_incomes = train_incomes + test_incomes

print("--- INCORRECT METHOD (Leakage) ---")
# Calculating mean over the entire dataset
leaked_mean = np.nanmean(all_incomes)
print(f"Leaked Mean: {leaked_mean}")
train_imputed_bad = [x if not np.isnan(x) else leaked_mean for x in train_incomes]
print(f"Imputed Training Data: {train_imputed_bad}")

print("\n--- CORRECT METHOD (No Leakage) ---")
# Calculating mean over training data ONLY
correct_mean = np.nanmean(train_incomes)
print(f"Correct Mean: {correct_mean}")
train_imputed_good = [x if not np.isnan(x) else correct_mean for x in train_incomes]
test_imputed_good = [x if not np.isnan(x) else correct_mean for x in test_incomes]
print(f"Imputed Training Data: {train_imputed_good}")
print(f"Imputed Test Data:     {test_imputed_good}")


### 6. Finance Perspective
In algorithmic trading, data leakage is the number one cause of failed hedge funds. If your backtest scales today's stock price using the maximum price of the entire year (which includes tomorrow's prices), your trading model is effectively predicting the past using the future. It will look like a money-printing machine in the backtest but will lose everything in live markets.

---

# Part 2: Basic Data Cleaning
---
Basic data cleaning corrects invalid, redundant, or missing observations before modeling. We start with the most foundational steps: removing zero-variance features and duplicate rows.

## 2.1 Zero-Variance Features

### 1. Overview
A feature whose values are identical across all observations contains zero information for separating or predicting observations. It is completely redundant and must be removed to save computational memory and prevent multicollinearity warnings.

### 2. Concept & Formula
**Population Variance Formula**:
$$ Var(X) = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2 $$
If every $x_i = c$ (a constant), then the mean $\mu = c$, and every squared deviation is zero. Thus, $Var(X) = 0$.

**Decision Rule**: Remove a feature when its variance is exactly zero, or below a very small threshold chosen for the specific modeling problem.

### 3. Simple Explanation
If you are trying to predict which loan applicants will default, and one of your columns is `country_of_origin` where every single applicant is from "Vietnam" (VN), that column doesn't help you distinguish a good borrower from a bad borrower. It's noise.

### 4. Manual Calculation
**Toy Dataset: SME Credit Portfolio**
Let's consider a feature `branch_code` with values for 5 applicants: $b = (1, 1, 1, 1, 1)$.
1. Mean: $\bar{b} = 1$.
2. Variance:
$$ Var(b) = \frac{(1-1)^2 + (1-1)^2 + (1-1)^2 + (1-1)^2 + (1-1)^2}{5} = \frac{0}{5} = 0 $$
Therefore, `branch_code` should be removed.

Compare this with `loan_amount`: $x = (100, 150, 120, 200, 150)$.
1. Mean: $\bar{x} = 144$.
2. Sum of squared deviations: $\sum (x_i - 144)^2 = 5720$.
3. Variance: $Var(x) = \frac{5720}{5} = 1144$.
Loan amount varies significantly and is retained.

### 5. Code Python


In [ ]:
# Toy Dataset
csv_credit = """loan_amount,interest_rate,debt_ratio,branch_code,default_flag
100,8.5,0.35,1,0
150,9.2,0.48,1,1
120,8.8,0.40,1,0
200,10.1,0.55,1,1
150,9.2,0.48,1,1"""

df_credit = pd.read_csv(io.StringIO(csv_credit))
print("Original Dataset:")
print(df_credit)

# Tool: sklearn.feature_selection.VarianceThreshold
X_credit = df_credit.drop(columns=['default_flag'])
selector = VarianceThreshold(threshold=0.0) # Remove exact constants
X_selected = selector.fit_transform(X_credit)

retained_cols = selector.get_feature_names_out(input_features=X_credit.columns)
print(f"\nRetained Columns: {retained_cols}")

# For categorical features, use nunique()
country = pd.Series(['VN'] * 5)
print(f"\nNumber of unique categories in 'country': {country.nunique()}")
if country.nunique() == 1:
    print("Action: Remove 'country' feature because it is constant.")


### 6. Finance Perspective
When banks merge datasets from different regional systems, they often pull metadata columns like `currency_code` (e.g., always "USD") or `system_version` (always "v2.0"). Feeding hundreds of these zero-variance columns into a model drastically increases training time and can cause numerical instability in matrix-inversion algorithms used in portfolio optimization or linear regression.

---

## 2.2 Duplicate Applications

### 1. Overview
Repeated applications or duplicate rows can overweight one specific borrower profile, causing the model to learn biased distributions. More dangerously, duplicates can leak the exact same case across training and test sets.

### 2. Concept & Formula
Two rows $X_i$ and $X_j$ are considered duplicates if all their feature values are exactly identical: $X_{i, k} = X_{j, k}$ for all $k$ features. The decision rule is typically to retain the first occurrence (`keep='first'`) and drop the rest.

### 3. Simple Explanation
If a customer accidentally clicks the "Submit Loan Application" button twice on the bank's website, two identical rows are created. If we keep both, our model thinks there are two identical risky people instead of one, shifting the statistical reality.

### 4. Manual Calculation
Look at the toy dataset above:
- Row 1: `(150, 9.2, 0.48, 1, 1)`
- Row 4: `(150, 9.2, 0.48, 1, 1)`
All values match perfectly. Therefore, we retain row 1 and discard row 4.

### 5. Code Python


In [ ]:
# Pandas tool for duplicates
num_duplicates = df_credit.duplicated().sum()
print(f"Number of duplicate applications found: {num_duplicates}")

# Remove duplicates
credit_no_dup = df_credit.drop_duplicates(keep='first')
print("\nDataset after dropping duplicates:")
print(credit_no_dup)


### 6. Finance Perspective
In credit card fraud detection, fraudsters often use automated bots to blast thousands of identical applications simultaneously in hopes one gets approved. If we don't deduplicate data, the training set becomes highly imbalanced with artificial synthetic profiles, warping the model's understanding of true human behavior.

---

# Part 3: Handling Outliers
---
Anomalies or Outliers are observations that are significantly different from other observations in the dataset. They arise from incorrect transaction records, data-entry errors, or genuine market shocks. Outliers can skew statistical summaries and disproportionately influence model parameters (especially in regression).

## 3.1 Standard Deviation Method

### 1. Overview
This is a parametric outlier detection method. It assumes that the data follows a Gaussian (Normal) distribution. It flags observations that lie extremely far from the central mean.

### 2. Concept & Formula
The **Z-Score** measures how many standard deviations away a point is from the mean.
$$ Z = \frac{x_i - \mu}{\sigma} $$
We flag an observation as an outlier if its absolute Z-score exceeds a threshold, typically $3\sigma$ (covering 99.7% of data in a perfect normal distribution).
Bounds: $[\mu - 3\sigma, \mu + 3\sigma]$

### 3. Simple Explanation
Imagine measuring the height of students. The average is 1.7m, and the standard deviation is 0.1m. A 3-sigma rule means anyone shorter than 1.4m or taller than 2.0m is considered an outlier.

### 4. Manual Calculation
**Toy target values (Housing MEDV)**:
`MEDV = (24, 21.6, 34.7, 33.4, 36.2, 500)`
Notice the last value `500` is incredibly large.
1. Mean: $\bar{x} = 108.32$
2. Sample standard deviation: $s = 191.98$
3. Two-standard-deviation bounds ($2\sigma$ used for this small toy sample):
   - Lower limit: $L = 108.32 - 2(191.98) = -275.64$
   - Upper limit: $U = 108.32 + 2(191.98) = 492.27$
Since $500 > 492.27$, the final observation is flagged.

**Caution**: Notice how the single outlier `500` inflated the mean from ~30 all the way to `108.32`! This is the primary weakness of the standard deviation method.

### 5. Code Python


In [ ]:
# Toy Housing Dataset with anomaly
csv_housing = """RM,MEDV
6.575,24.0
6.421,21.6
7.185,34.7
6.998,33.4
7.147,36.2
12.0,500.0""" # Extreme anomaly injected here

df_housing = pd.read_csv(io.StringIO(csv_housing))
print("Initial Housing Data:")
print(df_housing.to_string(index=False))

data_col = df_housing['MEDV']
mean_val = data_col.mean()
std_val = data_col.std()
cut_off = std_val * 2  # Using 2 sigma for toy example

lower = mean_val - cut_off
upper = mean_val + cut_off

print(f"\nMean: {mean_val:.2f}, Std: {std_val:.2f}")
print(f"Bounds: [{lower:.2f}, {upper:.2f}]")

outliers_std = df_housing[(data_col < lower) | (data_col > upper)]
print(f"\nFound {len(outliers_std)} outliers using Std Dev method:")
print(outliers_std.to_string(index=False))


### 6. Finance Perspective
The Standard Deviation method is deeply flawed for financial returns because market returns are notoriously "fat-tailed" (leptokurtic), not normally distributed. A 3-sigma event in a normal distribution happens once every 1,000 days. In the real stock market, 3-sigma events (like sudden crashes) happen several times a year! Using Z-scores in finance often leads to over-flagging genuine market dynamics as "errors".

---

## 3.2 Interquartile Range (IQR) Method

### 1. Overview
The IQR method is a robust, non-parametric method. It does not assume a Gaussian distribution and is highly resistant to the outliers themselves.

### 2. Concept & Formula
The Interquartile Range (IQR) measures the spread of the middle 50% of the data.
$$ IQR = Q_3 - Q_1 $$
Where $Q_1$ is the 25th percentile and $Q_3$ is the 75th percentile.
Fences (Bounds) are calculated as:
$$ \text{Lower} = Q_1 - 1.5 \times IQR $$
$$ \text{Upper} = Q_3 + 1.5 \times IQR $$

### 3. Simple Explanation
Instead of using the mean (which gets dragged by extremes), we line up all values from smallest to largest and find the middle block. We define normal boundaries relative to this sturdy middle block. Because a massive outlier is just "the last guy in line", it doesn't shift the position of the middle block at all.

### 4. Manual Calculation
Using the same MEDV values: `(21.6, 24.0, 33.4, 34.7, 36.2, 500)`
1. Calculate quartiles (using Pandas linear interpolation method):
   - $Q_1 = 26.35$
   - $Q_3 = 35.825$
2. Calculate IQR:
   - $IQR = 35.825 - 26.35 = 9.475$
3. Outlier Fences:
   - $L = 26.35 - 1.5(9.475) = 12.1375$
   - $U = 35.825 + 1.5(9.475) = 50.0375$
Because $500 > 50.0375$, it is flagged as an outlier. Notice that the boundaries ($12$ to $50$) make much more sense for the normal data than the Z-score boundaries ($-275$ to $492$).

### 5. Code Python


In [ ]:
# IQR Implementation
Q1 = data_col.quantile(0.25)
Q3 = data_col.quantile(0.75)
IQR = Q3 - Q1

lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Robust Bounds: [{lower_iqr:.2f}, {upper_iqr:.2f}]")

outliers_iqr = df_housing[(data_col < lower_iqr) | (data_col > upper_iqr)]
print(f"\nFound {len(outliers_iqr)} outliers using IQR method:")
print(outliers_iqr.to_string(index=False))


### 6. Finance Perspective
IQR is the gold standard for robust univariate screening in finance. When cleaning fundamental accounting data (like P/E ratios across 500 companies), using IQR ensures that a few crazy tech companies with P/E ratios of 10,000 don't ruin the statistical thresholds used to evaluate normal companies.

---

## 3.3 Local Outlier Factor (LOF)

### 1. Overview
Standard Deviation and IQR are univariate methods (looking at one column at a time). But what if a point is an outlier only when considering the *relationship* between multiple columns? Local Outlier Factor (LOF) is an unsupervised, model-based local-density method useful for identifying multivariate outliers.

### 2. Concept & Formula
LOF compares the local density of a given data point to the local densities of its $k$ nearest neighbors.
If a point has a much lower density than its neighbors, it is considered an isolated observation (an outlier).
- **Reachability Distance**: Distance to the $k$-th neighbor.
- **Local Reachability Density (LRD)**: Inverse of the average reachability distance to neighbors.
- **LOF Score**: The ratio of the average LRD of neighbors to the point's own LRD.
  $$ LOF(D) = \frac{\frac{1}{k} \sum LRD(\text{neighbors})}{LRD(D)} $$
A substantially larger LOF score (>> 1) indicates an outlier.

### 3. Simple Explanation
Imagine a bustling city center (dense cluster) and a farmhouse far out in the countryside (isolated point). A person in the city center has many neighbors very close by. The farmhouse has neighbors that are very far away. LOF measures exactly this: "Are you much lonelier than your neighbors are?"

### 4. Manual Calculation
Consider 1D points: $A = 0.9, B = 1.0, C = 1.1$, and a distant point $D = 5.0$, with $k=2$.
For $D$, the two nearest neighbors are $C$ and $B$.
1. Distances: $d(D, C) = |5.0 - 1.1| = 3.9$, $d(D, B) = |5.0 - 1.0| = 4.0$.
2. Average reachability distance for $D$: $(3.9 + 4.0) / 2 = 3.95$.
3. Local reachability density for $D$: $LRD(D) = 1 / 3.95 \approx 0.253$.
Assuming we calculated $LRD(C) = 6.667$ and $LRD(B) = 5.000$ (since they are tightly packed).
4. LOF Score for $D$:
$$ LOF(D) = \frac{(6.667 / 0.253) + (5.000 / 0.253)}{2} \approx 23.04 $$
Because $23.04 \gg 1$, point $D$ is strongly flagged as a local outlier.

### 5. Code Python


In [ ]:
# Adding a subtle multivariate outlier
# Normal houses: 6 rooms -> $20-30k. 
# Outlier house: 6 rooms -> $500k! (Not an outlier in RM alone, but an outlier in the RM-MEDV relationship)
from sklearn.preprocessing import StandardScaler

X_lof = df_housing[['RM', 'MEDV']]
X_scaled = StandardScaler().fit_transform(X_lof)

# Tuning parameter: n_neighbors
lof = LocalOutlierFactor(n_neighbors=2) 
yhat = lof.fit_predict(X_scaled) # returns -1 for outliers and 1 for inliers

mask = (yhat == -1)
print(f"Found {sum(mask)} multivariate outliers using LOF:")
print(df_housing[mask].to_string(index=False))

# Warning: Never fit LOF on combined train/test data before splitting!


### 6. Finance Perspective
Multivariate outliers are incredibly important in credit card fraud. A transaction of $500 might not be a univariate outlier (many people spend $500). A transaction in Nigeria might not be an outlier. But a $500 transaction in Nigeria from an account that just bought a $2 coffee in New York 10 minutes ago is a massive multivariate outlier. LOF captures these relational anomalies perfectly.

---

# Part 4: Handling Missing Data
---
Most machine learning algorithms (like SVMs, Neural Networks, and Scikit-learn's default Random Forests) cannot perform matrix multiplication on `NaN` values. The treatment strategy can materially affect model performance.

## 4.1 Missing Data Removal

### 1. Overview
The simplest approach is Complete-case analysis: removing any observation (row) that contains one or more missing values, or dropping entirely broken columns.

### 2. Concept & Formula
- **Row Removal (`dropna(axis=0)`)**: Discard $X_i$ if $\exists j$ such that $X_{i,j}$ is `NaN`.
- **Column Removal (`dropna(axis=1)`)**: Discard feature $F_j$ if missingness exceeds a threshold $\theta$ (e.g., 50%).

### 3. Simple Explanation
If a survey respondent skipped a question, you just throw away their entire survey. 

### 4. Manual Calculation
Suppose we have 5 loan applications:
- Row 0: 0 missing
- Row 1: 1 missing
- Row 2: 1 missing
- Row 3: 0 missing
- Row 4: 1 missing
Complete rows = 2/5 = 40%. Removing incomplete rows forces us to discard 60% of our precious sample, introducing massive bias if the missingness wasn't completely random.

### 5. Code Python


In [ ]:
df_loans = pd.DataFrame({
    'income': [60, 75, 80, 90, 70],
    'credit_score': [720, np.nan, 680, 650, 700]
})

print("Original Data:")
print(df_loans)

# Auditing
print("\nMissing values per column:")
print(df_loans.isnull().sum())

# Dropping
data_dropped = df_loans.dropna(axis=0)
print("\nData after dropping missing rows:")
print(data_dropped)


### 6. Finance Perspective
In Macroeconomics, if you drop an entire month of global data just because one minor emerging market failed to report its inflation rate, you destroy your time series continuity. Row dropping is almost always discouraged in finance unless the missingness is trivial (< 1% of data).

---

## 4.2 Statistical Imputation

### 1. Overview
Replaces a missing value with a statistical summary of that feature, such as the mean, median, or mode.

### 2. Concept & Formula
$$ \hat{x}_{i,j} = \mu_j \quad \text{or} \quad \hat{x}_{i,j} = \text{median}(F_j) $$
**Trade-off**: It is exceptionally simple and fast, but it heavily reduces the feature's variance and completely ignores relationships with other features.

### 3. Simple Explanation
If we don't know someone's credit score, we just give them the average credit score of the entire population.

### 4. Manual Calculation
**Mean Imputation for `credit_score`:**
Observed values: `(720, 680, 650, 700)`.
$$ \bar{x} = \frac{720 + 680 + 650 + 700}{4} = 687.5 $$
The missing credit score in Row 1 becomes 687.5.

### 5. Code Python


In [ ]:
imputer_mean = SimpleImputer(strategy='mean')
data_imputed_mean = imputer_mean.fit_transform(df_loans)

print("Mean Imputation Results:")
print(pd.DataFrame(data_imputed_mean, columns=df_loans.columns))


### 6. Finance Perspective
Mean imputation is dangerous for risk-sensitive variables. If missing credit scores generally belong to rejected, low-quality borrowers (MNAR), imputing them with the population average ($687.5$) will artificially elevate their creditworthiness, exposing the bank to massive uncalculated risk.

---

## 4.3 KNN Imputation

### 1. Overview
A multivariate method that finds the $k$ most similar rows and imputes a missing value using the neighbors' average. It preserves relationships between variables much better than mean imputation.

### 2. Concept & Formula
To impute $\hat{x}_{i, j}$:
1. Compute Euclidean distance between row $i$ and all other complete rows based on the available features.
2. Select the top $k$ rows with the smallest distance.
3. Take the weighted or uniform average of feature $j$ from those $k$ rows.

**Practical requirement**: You MUST scale features first when their units differ (e.g., Income in thousands, Credit Score in hundreds); otherwise, large-scale variables dominate the distance math.

### 3. Simple Explanation
Instead of giving a missing student the class average, we look at their other grades (e.g., High in English, Low in History). We find 2 other students with similar English and History grades, and average their Math scores to guess the missing Math score.

### 4. Manual Calculation
Impute credit score for Applicant A (Income = 75, Credit = `NaN`).
Nearest complete neighbors:
- Applicant B: (Income = 70, Credit = 700)
- Applicant C: (Income = 80, Credit = 680)
Distance to B: $|75 - 70| = 5$. Distance to C: $|75 - 80| = 5$.
With $k=2$ and uniform weights, the imputed score is:
$$ \widehat{score} = \frac{700 + 680}{2} = 690 $$

### 5. Code Python


In [ ]:
knn_imputer = KNNImputer(n_neighbors=2, weights='uniform')
data_imputed_knn = knn_imputer.fit_transform(df_loans)

print("KNN Imputation Results:")
print(pd.DataFrame(data_imputed_knn, columns=df_loans.columns))


### 6. Finance Perspective
KNN Imputation is highly favored in quantitative finance when building cross-sectional equity models. If a mid-cap tech company is missing its R&D expenditure data, KNN will intelligently estimate it based on other mid-cap tech companies with similar revenue profiles, preserving the delicate financial ratios.

---

## 4.4 Iterative Imputation

### 1. Overview
An advanced strategy that treats imputation as a machine-learning problem itself, modeling each incomplete feature as a function of all the other features.

### 2. Concept & Formula
**Iterative logic**:
1. Initialize missing entries with mean imputation.
2. Predict one incomplete feature ($Y$) from all the others ($X$) using a regressor (like Ridge Regression or Random Forest).
3. Update the missing values with the predictions.
4. Repeat for all features, cyclically, until estimates stabilize (convergence).

### 3. Simple Explanation
It's like a Sudoku puzzle. You make a rough guess for all empty boxes. Then you look at Box 1 and recalculate it based on all surrounding boxes. Then you do Box 2. By the time you finish the board, your guesses get sharper and sharper. You repeat this until the numbers stop changing.

### 4. Manual Calculation
**One simplified regression step**:
Suppose complete pairs of (income, loan_amount) are `(60, 100), (80, 140), (100, 180)`.
The fitted linear relationship is exactly:
$$ \widehat{loan} = 2 \times (\text{income}) - 20 $$
For an applicant with income 75 and a missing loan amount, Iterative Imputation predicts:
$$ \widehat{loan} = 2(75) - 20 = 130 $$

### 5. Code Python


In [ ]:
# Iterative Imputer is experimental, enable it first
iter_imputer = IterativeImputer(max_iter=10, random_state=0)
data_imputed_iter = iter_imputer.fit_transform(df_loans)

print("Iterative Imputation Results:")
print(pd.DataFrame(data_imputed_iter, columns=df_loans.columns))


### 6. Finance Perspective
Iterative Imputation (often implemented via the MICE algorithm in R or Scikit-learn) is heavily used by stress-testing departments in banks (e.g., CCAR in the US) to fill in historical gaps in macroeconomic variables by learning the complex web of correlations between GDP, unemployment, and interest rates.

---

# Part 5: Automating with Pipelines
---

## 5.1 Scikit-learn Pipelines

### 1. Overview
To completely prevent the Data Leakage discussed in Part 1, we must encapsulate preprocessing (imputation, scaling) and modeling into a single, cohesive workflow object. 

### 2. Concept & Formula
`sklearn.pipeline.Pipeline` chains transformers and a final estimator into one object.
During `fit()`, it automatically calls `fit_transform()` on intermediate steps, learning only from the training data.
During `predict()`, it strictly calls `transform()` on the test data, ensuring no test data statistics leak into the formulas.

### 3. Simple Explanation
A Pipeline is like an assembly line in a factory. The raw material (data) enters step 1 (cleaning), moves to step 2 (imputation), and finishes at step 3 (modeling). The assembly line machinery is strictly calibrated ONLY using yesterday's materials (training data). When today's fresh materials (test data) arrive, they just pass through the fixed machinery.

### 4. Manual Calculation / Architecture
Pipeline Architecture:
1. `train_test_split`: Splits data.
2. `SimpleImputer`: Handles missing values first, ensuring complete inputs.
3. `RandomForestClassifier`: Builds trees and aggregates outputs.
All bound together in `Pipeline(steps=[('imputer', ...), ('model', ...)])`.

### 5. Code Python


In [ ]:
# Toy setup for Pipeline
X = pd.DataFrame({'income': [60, 75, 80, np.nan, 70, 90, 85, np.nan, 65, 100]})
y = np.array([0, 0, 1, 1, 0, 1, 1, 0, 0, 1])

# 1. SPLIT FIRST (Golden Rule)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Define Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(random_state=42, n_estimators=10))
])

# 3. Fit Pipeline ONLY on training data
pipeline.fit(X_train, y_train)
print("Pipeline trained successfully (Learned median from X_train ONLY).")

# 4. Predict on Test data (Uses the training median to fill X_test NaNs safely)
y_pred = pipeline.predict(X_test)
print(f"Accuracy on unseen test set: {accuracy_score(y_test, y_pred):.4f}")


### 6. Finance Perspective
In production trading environments, models must score live, streaming market data tick-by-tick. Without a Pipeline object that securely holds the historical scaling parameters and imputation bounds, deploying an ML model to production is a software engineering nightmare. Pipelines ensure that your research code translates 1:1 into production code.

---

# Part 6: Practice & Exercises
---

## Lab 1: Credit-Risk Data Integrity Audit

**Tasks**:
1. Load the German Credit dataset from OpenML.
2. Identify constant numerical and categorical features.
3. Count and remove duplicate applications.
4. Report the portfolio shape before and after cleaning.



In [ ]:
# Lab 1 Setup
# credit_g = fetch_openml(name='credit-g', version=1, as_frame=True)
# df_lab1 = credit_g.frame

# ==========================================================
# YOUR CODE HERE
# ==========================================================
# print(df_lab1.shape)
pass
